# 💼 ETF Ki Dukan Strategy -

A trading system to earn per day profit. **You will buy everyday and will earn everyday.**

👉 Conditions:
- Volume should be greater than 10000
- Remove all Bond/Liquid ETF.

- (*) Make high priority of those ETFs which are down by 3% on daily candle

👍 Pros:
- We can easily average out ETF if market is in Bear phase.
- ETF creates a good portfolio and safer investment.

👎 Cons:
- If ETF's price movement is very less then, we can't achieve minimum profit booking easily like, Liquid ETF.


##### Ref:
1. ETF से शेयरों की दुकान Part 1: https://www.youtube.com/watch?v=IN0IH_S3d7k
2. ETF की दुकान स्ट्रेटेजी ETF Trading and Investing Strategies Part 2: https://www.youtube.com/watch?v=1UJNwvBNKXk
3. 

##### Credits: 
🙏🙏🙏 Thanks to Mahesh Kaushik Sir 🙏🙏🙏



# Python Script

In [ ]:
import requests
import pandas as pd
import datetime
import time
import json
import asyncio
import aiohttp


base_nse_url = "https://www.nseindia.com/"
nse_headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}


# common nse api fetch functionality
def nse_fetch_data(nse_url: str) -> dict:
    """
    Fetch data from NSE API.

    :param nse_url: URL for the NSE API endpoint
    :type nse_url: str

    :return: JSON response from the NSE API
    :rtype: dict
    """
    nse_session = requests.Session()
    nse_session.headers.update(nse_headers)
    nse_session.get(base_nse_url, headers=nse_headers,  timeout=10)
    nse_session.get(base_nse_url+"/option-chain", headers=nse_headers,  timeout=10)

    full_nse_api_url = base_nse_url + nse_url
    # print(f"calling {full_nse_api_url} ..")
    response = nse_session.get(full_nse_api_url)
    output = response.json()
    return output


# get all nse etfs
def get_nse_etfs():
    etf_data_url = "api/etf"
    etf_data = nse_fetch_data(nse_url=etf_data_url)
    etf_data_df = pd.DataFrame(etf_data.get('data'))

    # 'symbol', 'assets', 'open', 'high', 'low', 'ltP', 'chn', 'per', 'qty', 'trdVal', 'nav', 'wkhi', 'wklo', 'prevClose', 
    # 'stockIndClosePrice', 'perChange365d', 'perChange30d', 'date365dAgo', 'date30dAgo', 'ypc',
    # 'mpc', 'xdt', 'cact', 'nearWKH', 'nearWKL', 'chartTodayPath', 'chart30dPath', 'chart365dPath', 'series', 'meta'

    # remove unnecessary columns
    etf_data_df = etf_data_df.loc[:, ['symbol', 'assets', 'open', 'high', 'low', 'ltP', 'chn', 'per', 'qty', 'nav']]

    # filter the etfs by, volumn > 10000
    etf_data_df['qty'] = etf_data_df['qty'].astype(float)
    etf_data_df = etf_data_df.loc[etf_data_df['qty'] >= 10000]
        
    # filter out bond etfs
    search_values_for_debt_etf = ['LIQUID', 'BOND', 'G-SEC', 'GSEC', 'GILT']
    pattern = '|'.join(search_values_for_debt_etf)
    etf_data_df = etf_data_df[~etf_data_df['assets'].str.contains(pattern, case=False, na=False)]

    # rename the columns
    etf_data_df.rename(columns={"symbol": "Symbol", "assets": "Assets", "open": "Open", "high": "High", "low": "Low",
                                "ltP": "Close", "chn": "Chng", "per": "%Chng", "qty": "Volume", "nav": "iNAV"},
                                inplace=True)
    
    return etf_data_df.reset_index(drop=True)


# get 20DMA, 52Wk High & Low data (*not date) from last 31 days daily candle data 
async def _get_nse_historical_trade_data(symbols, from_date, to_date):
    nse_historical_trade_data_url = 'api/NextApi/apiClient/GetQuoteApi?functionName=getHistoricalTradeData&symbol={}&series=EQ&fromDate={}&toDate={}'
    results = []
    tasks = []
    async with aiohttp.ClientSession() as nse_session:
        for symbol in symbols:
            full_nse_url = base_nse_url + nse_historical_trade_data_url.format(symbol, from_date, to_date)
            # print(f"NSE Api: {full_nse_url} has been called ..")
            tasks.append(nse_session.get(full_nse_url, headers=nse_headers, ssl=False))
        responses = await asyncio.gather(*tasks)
        for response in responses:
            results.append(await response.json())
    df_list = []
    for i, res in enumerate(results):
        out_json = res
        out_df = pd.DataFrame(out_json)
        out_df['mtimestamp'] = pd.to_datetime(out_df['mtimestamp'], format='%d-%b-%Y')   # format='%Y-%m-%d'
        out_df = out_df.sort_values('mtimestamp', ascending=True)
        out_df['20DMA'] = out_df['chLastTradedPrice'].rolling(window=20).mean()
        df_list.append(out_df.tail(1))
    df = pd.concat(df_list, ignore_index=True) 

    # 'chSymbol', 'chSeries', 'chPreviousClsPrice', 'chOpeningPrice','chTradeHighPrice', 'chTradeLowPrice', 'chLastTradedPrice','chClosingPrice', 'vwap', 'chTotTradedQty', 'chTotTradedVal','chTotalTrades', 'ch52WeekHighPrice', 'ch52WeekLowPrice', 'mtimestamp'
    df = df.loc[:, ['mtimestamp', 'chSymbol', 'chOpeningPrice', 'chTradeHighPrice', 'chTradeLowPrice', 'chLastTradedPrice', 'vwap', 
                    '20DMA', 'chTotTradedQty', 'chTotalTrades', 'ch52WeekHighPrice', 'ch52WeekLowPrice']]
    df.rename(columns={'mtimestamp': 'Date', 'chSymbol': 'Symbol', 'chOpeningPrice': 'Open', 'chTradeHighPrice': 'High', 
                       'chTradeLowPrice': 'Low', 'chLastTradedPrice': 'Close', 'vwap': 'VWAP', 'chTotTradedQty': 'Volume', 
                       'chTotalTrades': 'Total Trades', 'ch52WeekHighPrice': '52wk High', 'ch52WeekLowPrice': '52wk Low'},
                       inplace=True)
    return df


# get all time high low data and date
async def _get_nse_alltime_high_low(symbols):
    nse_alltime_high_low_url = 'api/NextApi/apiClient/GetQuoteApi?functionName=getHistoricalPeriodic52WeekHighLow&symbol={}'
    results = []
    tasks = []
    async with aiohttp.ClientSession() as nse_session:
        for symbol in symbols:
            full_nse_url = base_nse_url + nse_alltime_high_low_url.format(symbol)
            # print(f"NSE Api: {full_nse_url} has been called ..")
            tasks.append(nse_session.get(full_nse_url, headers=nse_headers, ssl=False))
        responses = await asyncio.gather(*tasks)
        for response in responses:
            results.append(await response.json())
    df_list = []
    for i, res in enumerate(results):
        out_json = res.get('data')
        out_df = pd.DataFrame(out_json, index=[0])
        out_df['Symbol'] = symbols[i]
        out_df['maxDate'] = pd.to_datetime(out_df['maxDate'], format='%d-%b-%Y') 
        out_df['minDate'] = pd.to_datetime(out_df['minDate'], format='%d-%b-%Y') 
        df_list.append(out_df)
    df = pd.concat(df_list, ignore_index=True)
    df.rename(columns={'max': 'ATH', 'maxDate': 'ATH Dt', 'min': 'ATL', 'minDate': 'ATL Dt'}, inplace=True)
    return df.loc[:, ['Symbol', 'ATH', 'ATH Dt', 'ATL', 'ATL Dt']]


async def _get_nse_52wk_high_low(symbols):
    # api/NextApi/apiClient/GetQuoteApi?functionName=getHistoricalPeriodicData&symbol={symbol}&type=weekly52
    # api/NextApi/apiClient/GetQuoteApi?functionName=getHistoricalPeriodicData&symbol={symbol}&year=2026&type=yearly
    # api/NextApi/apiClient/GetQuoteApi?functionName=getHistoricalPeriodicData&symbol={symbol}&year=2026&month=02&type=monthly
    nse_52wk_high_low_url = 'api/NextApi/apiClient/GetQuoteApi?functionName=getHistoricalPeriodicData&symbol={}&type=weekly52'
    results = []
    tasks = []
    async with aiohttp.ClientSession() as nse_session:
        for symbol in symbols:
            full_nse_url = base_nse_url + nse_52wk_high_low_url.format(symbol)
            # print(f"NSE Api: {full_nse_url} has been called ..")
            tasks.append(nse_session.get(full_nse_url, headers=nse_headers, ssl=False))
        responses = await asyncio.gather(*tasks)
        for response in responses:
            results.append(await response.json())
    df_list = []
    for i, res in enumerate(results):
        data = dict()
        data['Symbol'] = symbols[i]
        data['52Wk High Dt'] = res.get('high').get('high_price_date')
        data['52Wk Low Dt'] = res.get('low').get('low_price_date')
        out_df = pd.DataFrame(data, index=[0])
        out_df['52Wk High Dt'] = pd.to_datetime(out_df['52Wk High Dt'], format='%d-%b-%Y') 
        out_df['52Wk Low Dt'] = pd.to_datetime(out_df['52Wk Low Dt'], format='%d-%b-%Y') 
        df_list.append(out_df)
    df = pd.concat(df_list, ignore_index=True)
    return df.loc[:, ['Symbol', '52Wk High Dt', '52Wk Low Dt']]

In [ ]:
etf_data_df = get_nse_etfs()
unique_symbols = etf_data_df['Symbol'].unique().tolist()
    
cur_date_obj = datetime.datetime.today()
from_date_obj = cur_date_obj - datetime.timedelta(days=31)
from_date = from_date_obj.strftime("%d-%m-%Y")
to_date = cur_date_obj.strftime("%d-%m-%Y")

nse_etf_hist_trade_df = await _get_nse_historical_trade_data(unique_symbols, from_date, to_date)

# Join: current open-high-low-close, 20DMA
etf_ohlc_20dma_df = pd.merge(etf_data_df, 
                    nse_etf_hist_trade_df.loc[:,['Date', 'Symbol', 'VWAP', '20DMA', 'Total Trades', '52wk High', '52wk Low']], 
                    left_on=['Symbol'], right_on=['Symbol'], how='inner')

time.sleep(2)
# All time High Low data with date
nse_etf_alltime_high_low_df = await _get_nse_alltime_high_low(unique_symbols)

# Join: current open-high-low-close, 20DMA, All time High Low
etf_ohlc_20dma_athl_df = pd.merge(etf_ohlc_20dma_df, nse_etf_alltime_high_low_df, 
                             left_on=['Symbol'], right_on=['Symbol'], how='inner')

time.sleep(2)
# 52 Week High & Low Date
nse_etf_52week_high_low_date_df = await _get_nse_52wk_high_low(unique_symbols)

# Join: current open-high-low-close, 20DMA, All time High Low, 52 Week High & Low Date
etf_ohlc_20dma_athl_52wkhl_df = pd.merge(etf_ohlc_20dma_athl_df, nse_etf_52week_high_low_date_df, 
                                         left_on=['Symbol'], right_on=['Symbol'], how='inner')

print(f"final Dataframe Loaded: {etf_ohlc_20dma_athl_52wkhl_df.shape}")
